# System Health and State Transitions Monitoring

This notebook monitors the overall health of the ml_heating system, focusing on state transitions, control loop stability, and identifying potential anomalies in system behavior.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta, timezone

from src.analysis.data_loader import DataLoader

# Configure plotting
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_context("notebook")
plt.rcParams['figure.figsize'] = (12, 6)

## 1. Data Loading

Load recent operational data to analyze state transitions and system health.

In [ ]:
loader = DataLoader()
end_time = datetime.now(timezone.utc)
start_time = end_time - timedelta(days=7)

features = [
    'indoor_temperature',
    'ml_target_temperature',
    'heating_state',
    'control_loop_latency',
    'system_health_score'
]

try:
    df = loader.fetch_training_data(start_time=start_time, end_time=end_time, features=features)
    print(f"Loaded {len(df)} records from {start_time.strftime('%Y-%m-%d')} to {end_time.strftime('%Y-%m-%d')}")
except Exception as e:
    print(f"Error loading data: {e}")
    # Create dummy data for demonstration if InfluxDB is not available
    dates = pd.date_range(start=start_time, end=end_time, freq='5min')
    df = pd.DataFrame(index=dates)
    df['indoor_temperature'] = 20 + np.sin(np.linspace(0, 10, len(dates))) + np.random.normal(0, 0.1, len(dates))
    df['ml_target_temperature'] = 21
    df['heating_state'] = np.where(df['indoor_temperature'] < 20.5, 1, 0)
    df['control_loop_latency'] = np.random.normal(50, 10, len(dates))
    df['system_health_score'] = 100 - np.random.exponential(2, len(dates))
    print("Using generated dummy data for demonstration.")

## 2. State Transition Analysis

Analyze how frequently the system transitions between heating states (e.g., ON/OFF, different heating modes) to identify short-cycling or instability.

In [ ]:
if 'heating_state' in df.columns:
    # Calculate state transitions
    df['state_change'] = df['heating_state'].diff()
    transitions = df[df['state_change'] != 0].dropna(subset=['state_change'])
    
    print(f"Total state transitions in the period: {len(transitions)}")
    
    # Plot state over time
    fig, ax1 = plt.subplots(figsize=(14, 6))
    
    color = 'tab:red'
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Temperature (°C)', color=color)
    ax1.plot(df.index, df['indoor_temperature'], color=color, label='Indoor Temp')
    ax1.plot(df.index, df['ml_target_temperature'], color='tab:orange', linestyle='--', label='Target Temp')
    ax1.tick_params(axis='y', labelcolor=color)
    ax1.legend(loc='upper left')
    
    ax2 = ax1.twinx()  
    color = 'tab:blue'
    ax2.set_ylabel('Heating State', color=color)  
    ax2.fill_between(df.index, 0, df['heating_state'], color=color, alpha=0.3, step='post', label='Heating State')
    ax2.tick_params(axis='y', labelcolor=color)
    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(['OFF', 'ON'])
    
    fig.tight_layout()  
    plt.title('Temperature and Heating State Transitions')
    plt.show()
else:
    print("Heating state data not available.")

## 3. Control Loop Stability and Latency

Monitor the latency of the control loop to ensure timely responses and identify potential performance bottlenecks.

In [ ]:
if 'control_loop_latency' in df.columns:
    plt.figure(figsize=(12, 5))
    sns.histplot(df['control_loop_latency'].dropna(), bins=50, kde=True)
    plt.title('Control Loop Latency Distribution')
    plt.xlabel('Latency (ms)')
    plt.ylabel('Frequency')
    plt.axvline(df['control_loop_latency'].mean(), color='r', linestyle='dashed', linewidth=2, label=f"Mean: {df['control_loop_latency'].mean():.2f} ms")
    plt.legend()
    plt.show()
    
    # Time series of latency
    plt.figure(figsize=(14, 5))
    plt.plot(df.index, df['control_loop_latency'], alpha=0.7)
    plt.title('Control Loop Latency Over Time')
    plt.xlabel('Time')
    plt.ylabel('Latency (ms)')
    plt.show()
else:
    print("Control loop latency data not available.")

## 4. Overall System Health Score

Review the aggregated system health score over time.

In [ ]:
if 'system_health_score' in df.columns:
    plt.figure(figsize=(14, 5))
    plt.plot(df.index, df['system_health_score'], color='green')
    plt.fill_between(df.index, 0, df['system_health_score'], color='green', alpha=0.2)
    plt.title('System Health Score Over Time')
    plt.xlabel('Time')
    plt.ylabel('Health Score (0-100)')
    plt.ylim(0, 105)
    plt.axhline(90, color='orange', linestyle='--', label='Warning Threshold')
    plt.axhline(70, color='red', linestyle='--', label='Critical Threshold')
    plt.legend()
    plt.show()
else:
    print("System health score data not available.")